In [1]:
import sys

print(sys.executable)

D:\CODING\Modelling\.venv\Scripts\python.exe


In [8]:
import pandas as pd

occurrence = pd.read_csv("Occurance.csv")
conditions = pd.read_csv("Condition_Factors.csv")

print("Occurrence records:", len(occurrence))
print("Condition-factor records:", len(conditions))

Occurrence records: 136
Condition-factor records: 65535


In [9]:
print("OCCURRENCE COLUMNS:")
print(occurrence.columns.tolist())

print("\nCONDITION-FACTOR COLUMNS:")
print(conditions.columns.tolist())

OCCURRENCE COLUMNS:
['Longitude', 'Latitude', 'year', 'TaxonName', 'field_5', 'OBJECTID', 'Longitude_', 'Latitude_2', 'Presence']

CONDITION-FACTOR COLUMNS:
['OBJECTID', 'Longitude', 'Latitude', 'NDVI_1995', 'NDVI_2005', 'NDVI_2015', 'NDVI_2025', 'PCI_1995', 'PCI_2005', 'PCI_2015', 'PCI_2025', 'VHI_1995', 'VHI_2005', 'VHI_2015', 'VHI_2025', 'TCI_1995', 'TCI_2005', 'TCI_2015', 'TCI_2025', 'NDBI_1995', 'NDBI_2005', 'NDBI_2015', 'NDBI_2025', 'NDMI_1995', 'NDMI_2005', 'NDMI_2015', 'NDMI_2025', 'SAVI_1995', 'SAVI_2005', 'SAVI_2015', 'SAVI_2025', 'LST_1995', 'LST_2005', 'LST_2015', 'LST_2025', 'SMI_1995', 'SMI_2005', 'SMI_2015', 'SMI_2025', 'pop_1995', 'pop_2005', 'pop_2015', 'pop_2025', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'Nyeri_LULC_1995', 'Nyeri_LULC_2005', 'Nyeri_LULC_2015', 'Nyeri_LULC_2025', 'rain_1995', 'rain_2005', 'rain_2015', 'rain_2025']


In [10]:
print("OCCURRENCE DATA TYPES")
print(occurrence.dtypes)

print("\nCONDITION-FACTOR DATA TYPES")
print(conditions.dtypes)

print("\nOCCURRENCE RECORDS")
print(len(occurrence))

print("\nCONDITION-FACTOR RECORDS")
print(len(conditions))

OCCURRENCE DATA TYPES
Longitude     float64
Latitude      float64
year            int64
TaxonName         str
field_5           str
OBJECTID        int64
Longitude_    float64
Latitude_2    float64
Presence        int64
dtype: object

CONDITION-FACTOR DATA TYPES
OBJECTID           float64
Longitude          float64
Latitude           float64
NDVI_1995          float64
NDVI_2005          float64
NDVI_2015          float64
NDVI_2025          float64
PCI_1995           float64
PCI_2005           float64
PCI_2015           float64
PCI_2025           float64
VHI_1995           float64
VHI_2005           float64
VHI_2015           float64
VHI_2025           float64
TCI_1995           float64
TCI_2005           float64
TCI_2015           float64
TCI_2025           float64
NDBI_1995          float64
NDBI_2005          float64
NDBI_2015          float64
NDBI_2025          float64
NDMI_1995          float64
NDMI_2005          float64
NDMI_2015          float64
NDMI_2025          float64
SAVI_199

In [11]:
print("Occurrence OBJECTID examples:")
print(occurrence["OBJECTID"].head(10).to_list())

print("\nCondition-factor OBJECTID examples:")
print(conditions["OBJECTID"].head(10).to_list())

Occurrence OBJECTID examples:
[763, 763, 763, 763, 268, 1795, 1795, 1795, 3282, 2121]

Condition-factor OBJECTID examples:
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]


In [12]:
# Make working copies
occ = occurrence.copy()
factors = conditions.copy()

# Standardize OBJECTID
occ["OBJECTID"] = pd.to_numeric(occ["OBJECTID"], errors="coerce").astype("Int64")
factors["OBJECTID"] = pd.to_numeric(factors["OBJECTID"], errors="coerce").astype("Int64")

print("Occurrence OBJECTID type:", occ["OBJECTID"].dtype)
print("Condition-factor OBJECTID type:", factors["OBJECTID"].dtype)

print("\nOccurrence records:", len(occ))
print("Condition-factor records:", len(factors))

Occurrence OBJECTID type: Int64
Condition-factor OBJECTID type: Int64

Occurrence records: 136
Condition-factor records: 65535


In [13]:
missing_ids = occ.loc[
    ~occ["OBJECTID"].isin(factors["OBJECTID"]),
    "OBJECTID"
].unique()

print("Occurrence OBJECTIDs not found in condition factors:")
print(missing_ids)

print("\nNumber of missing OBJECTIDs:", len(missing_ids))

Occurrence OBJECTIDs not found in condition factors:
<IntegerArray>
[]
Length: 0, dtype: Int64

Number of missing OBJECTIDs: 0


In [14]:
merged = occ.merge(
    factors,
    on="OBJECTID",
    how="left",
    suffixes=("_occ", "_factor")
)

print("Merged records:", len(merged))
print("Merged columns:", len(merged.columns))

Merged records: 136
Merged columns: 65


In [15]:
years = [1995, 2005, 2015, 2025]

dynamic_variables = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "rain"
]

for var in dynamic_variables:
    merged[var] = pd.NA
    
    for year in years:
        source_column = f"{var}_{year}"
        mask = merged["year"] == year
        
        merged.loc[mask, var] = merged.loc[mask, source_column]

print("Year-specific environmental variables created successfully.")

Year-specific environmental variables created successfully.


In [16]:
merged["LULC"] = pd.NA

for year in years:
    source_column = f"Nyeri_LULC_{year}"
    mask = merged["year"] == year
    
    merged.loc[mask, "LULC"] = merged.loc[mask, source_column]

print("LULC variable created successfully.")

LULC variable created successfully.


In [17]:
merged[
    ["OBJECTID", "Longitude", "Latitude", "year",
     "NDVI", "PCI", "VHI", "TCI", "LST",
     "LULC", "rain"]
].head(10)

KeyError: "['Longitude', 'Latitude'] not in index"

In [18]:
print(merged.columns.tolist())

['Longitude_occ', 'Latitude_occ', 'year', 'TaxonName', 'field_5', 'OBJECTID', 'Longitude_', 'Latitude_2', 'Presence', 'Longitude_factor', 'Latitude_factor', 'NDVI_1995', 'NDVI_2005', 'NDVI_2015', 'NDVI_2025', 'PCI_1995', 'PCI_2005', 'PCI_2015', 'PCI_2025', 'VHI_1995', 'VHI_2005', 'VHI_2015', 'VHI_2025', 'TCI_1995', 'TCI_2005', 'TCI_2015', 'TCI_2025', 'NDBI_1995', 'NDBI_2005', 'NDBI_2015', 'NDBI_2025', 'NDMI_1995', 'NDMI_2005', 'NDMI_2015', 'NDMI_2025', 'SAVI_1995', 'SAVI_2005', 'SAVI_2015', 'SAVI_2025', 'LST_1995', 'LST_2005', 'LST_2015', 'LST_2025', 'SMI_1995', 'SMI_2005', 'SMI_2015', 'SMI_2025', 'pop_1995', 'pop_2005', 'pop_2015', 'pop_2025', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'Nyeri_LULC_1995', 'Nyeri_LULC_2005', 'Nyeri_LULC_2015', 'Nyeri_LULC_2025', 'rain_1995', 'rain_2005', 'rain_2015', 'rain_2025', 'NDVI', 'PCI', 'VHI', 'TCI', 'NDBI', 'NDMI', 'SAVI', 'LST', 'SMI', 'pop', 'rain', 'LULC']


In [19]:
merged[
    ["OBJECTID", "Longitude_occ", "Latitude_occ", "year",
     "NDVI", "PCI", "VHI", "TCI", "LST",
     "LULC", "rain"]
].head(10)

,OBJECTID,Longitude_occ,Latitude_occ,year,NDVI,PCI,VHI,TCI,LST,LULC,rain
0,763,37.000000,-0.633333,1995,0.547744,40.56507,49.27619,0.40443,27.570998,4.0,1591.478027
1,763,37.000000,-0.633330,1995,0.547744,40.56507,49.27619,0.40443,27.570998,4.0,1591.478027
2,763,37.000000,-0.633330,1995,0.547744,40.56507,49.27619,0.40443,27.570998,4.0,1591.478027
3,763,37.000000,-0.633330,1995,0.547744,40.56507,49.27619,0.40443,27.570998,4.0,1591.478027
4,268,37.133333,-0.316667,1995,0.816245,41.613003,73.779236,0.609076,19.859872,0.0,1201.499878
5,1795,36.831490,-0.347810,1995,0.811382,40.421284,71.112494,0.56091,21.674778,0.0,1270.514771
6,1795,36.831490,-0.347810,1995,0.811382,40.421284,71.112494,0.56091,21.674778,0.0,1270.514771
7,1795,36.831490,-0.347810,1995,0.811382,40.421284,71.112494,0.56091,21.674778,0.0,1270.514771
8,3282,36.666699,-0.416903,1995,0.727109,39.51131,76.710399,0.76245,14.080707,0.0,1812.491089
9,2121,36.935456,-0.420150,1995,0.580975,41.251748,55.642142,0.496424,24.10463,2.0,1178.557617


In [20]:
final_columns = [
    "OBJECTID",
    "Longitude_occ",
    "Latitude_occ",
    "year",
    "TaxonName",
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI",
    "LULC",
    "rain",
    "Presence"
]

final = merged[final_columns].copy()

# Rename the original GBIF coordinates
final = final.rename(columns={
    "Longitude_occ": "Longitude",
    "Latitude_occ": "Latitude"
})

print("Final records:", len(final))
print("Final columns:", len(final.columns))

print("\nColumns:")
print(final.columns.tolist())

Final records: 136
Final columns: 24

Columns:
['OBJECTID', 'Longitude', 'Latitude', 'year', 'TaxonName', 'NDVI', 'PCI', 'VHI', 'TCI', 'NDBI', 'NDMI', 'SAVI', 'LST', 'SMI', 'pop', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'LULC', 'rain', 'Presence']


In [21]:
final.head(10)

,OBJECTID,Longitude,Latitude,year,TaxonName,NDVI,PCI,VHI,TCI,NDBI,...,pop,SOC,Aspect,Slope,PH,TRI,TWI,LULC,rain,Presence
0,763,37.000000,-0.633333,1995,polyrhachis viscosa,0.547744,40.56507,49.27619,0.40443,0.190476,...,360.981494,42.948906,204.373871,13.032513,5.407969,0.623312,NaN,4.0,1591.478027,1
1,763,37.000000,-0.633330,1995,tetraponera parops,0.547744,40.56507,49.27619,0.40443,0.190476,...,360.981494,42.948906,204.373871,13.032513,5.407969,0.623312,NaN,4.0,1591.478027,1
2,763,37.000000,-0.633330,1995,tetraponera ambigua,0.547744,40.56507,49.27619,0.40443,0.190476,...,360.981494,42.948906,204.373871,13.032513,5.407969,0.623312,NaN,4.0,1591.478027,1
3,763,37.000000,-0.633330,1995,tetraponera gerdae,0.547744,40.56507,49.27619,0.40443,0.190476,...,360.981494,42.948906,204.373871,13.032513,5.407969,0.623312,NaN,4.0,1591.478027,1
4,268,37.133333,-0.316667,1995,Oligositoides semicinctum,0.816245,41.613003,73.779236,0.609076,0.396618,...,32.775685,48.015175,168.618957,10.947790,5.459834,0.676440,3.885907,0.0,1201.499878,1
5,1795,36.831490,-0.347810,1995,strumigenys katapelta,0.811382,40.421284,71.112494,0.56091,0.37178,...,0.098134,46.718647,119.377029,11.003533,5.300000,0.433929,2.850979,0.0,1270.514771,1
6,1795,36.831490,-0.347810,1995,hypoponera segnis,0.811382,40.421284,71.112494,0.56091,0.37178,...,0.098134,46.718647,119.377029,11.003533,5.300000,0.433929,2.850979,0.0,1270.514771,1
7,1795,36.831490,-0.347810,1995,discothyrea dryad,0.811382,40.421284,71.112494,0.56091,0.37178,...,0.098134,46.718647,119.377029,11.003533,5.300000,0.433929,2.850979,0.0,1270.514771,1
8,3282,36.666699,-0.416903,1995,"Dorylus molestus (Gerstacker, 1859) = Dorylus ...",0.727109,39.51131,76.710399,0.76245,0.215271,...,0.0,83.469872,174.910065,16.475666,5.220391,0.994688,2.494570,0.0,1812.491089,1
9,2121,36.935456,-0.420150,1995,"Anomma nigricans (Illiger, 1802) = Dorylus nig...",0.580975,41.251748,55.642142,0.496424,0.153151,...,2171.785656,32.265972,147.774490,10.435356,5.845007,0.810918,8.761285,2.0,1178.557617,1


In [22]:
missing = final.isna().sum()

print("Missing values by column:")
print(missing[missing > 0])

print("\nTotal missing values:", missing.sum())

Missing values by column:
VHI      4
TCI      4
LST      4
SMI      4
pop      1
TWI     30
rain    35
dtype: int64

Total missing values: 82


In [23]:
print("\nRecords containing any missing value:")
print(final[final.isna().any(axis=1)].to_string())


Records containing any missing value:
     OBJECTID  Longitude  Latitude  year                                                                    TaxonName      NDVI        PCI        VHI       TCI      NDBI      NDMI      SAVI        LST       SMI          pop        SOC      Aspect      Slope        PH       TRI       TWI LULC         rain  Presence
0         763  37.000000 -0.633333  1995                                                          polyrhachis viscosa  0.547744   40.56507   49.27619   0.40443  0.190476  0.190476  0.300664  27.570998   0.40443   360.981494  42.948906  204.373871  13.032513  5.407969  0.623312       NaN  4.0  1591.478027         1
1         763  37.000000 -0.633330  1995                                                           tetraponera parops  0.547744   40.56507   49.27619   0.40443  0.190476  0.190476  0.300664  27.570998   0.40443   360.981494  42.948906  204.373871  13.032513  5.407969  0.623312       NaN  4.0  1591.478027         1
2         763

In [24]:
missing_summary = final[final.isna().any(axis=1)][
    ["OBJECTID", "Longitude", "Latitude", "year", "TaxonName",
     "VHI", "TCI", "LST", "SMI", "pop", "TWI", "rain"]
]

missing_summary

,OBJECTID,Longitude,Latitude,year,TaxonName,VHI,TCI,LST,SMI,pop,TWI,rain
0,763,37.000000,-0.633333,1995,polyrhachis viscosa,49.27619,0.40443,27.570998,0.40443,360.981494,NaN,1591.478027
1,763,37.000000,-0.633330,1995,tetraponera parops,49.27619,0.40443,27.570998,0.40443,360.981494,NaN,1591.478027
2,763,37.000000,-0.633330,1995,tetraponera ambigua,49.27619,0.40443,27.570998,0.40443,360.981494,NaN,1591.478027
3,763,37.000000,-0.633330,1995,tetraponera gerdae,49.27619,0.40443,27.570998,0.40443,360.981494,NaN,1591.478027
10,2544,36.635714,-0.546953,1995,"Anomma nigricans (Illiger, 1802) = Dorylus nig...",77.878495,0.774465,13.627992,0.774465,0.0,NaN,1730.248047
11,4593,37.219867,-0.664578,1995,"Anomma nigricans (Illiger, 1802) = Dorylus nig...",43.099125,0.337624,30.088249,0.337624,207.084081,8.696597,NaN
16,2544,36.635714,-0.546953,1995,"Technomyrmex detorquens (Walker, 1859) = Techn...",77.878495,0.774465,13.627992,0.774465,0.0,NaN,1730.248047
17,4377,37.300000,-0.150000,1995,Apis mellifera,46.311789,0.801169,12.62179,0.801169,0.001029,1.378922,NaN
18,4774,37.308410,-0.152140,1995,"Boltonopone soror (Emery, 1899)",57.66723,0.981676,5.82021,0.981676,0.00023,1.478975,NaN
19,4774,37.333332,-0.166667,1995,"Belonogaster pennata Richards, 1982",57.66723,0.981676,5.82021,0.981676,0.00023,1.478975,NaN


In [25]:
print("Missing records by year:")
print(
    final[final.isna().any(axis=1)]
    .groupby("year")
    .size()
)

print("\nMissing values by year and variable:")
print(
    final.groupby("year")[["VHI", "TCI", "LST", "SMI", "pop", "TWI", "rain"]]
    .apply(lambda x: x.isna().sum())
)

Missing records by year:
year
1995    12
2005     8
2015     9
2025    14
dtype: int64

Missing values by year and variable:
      VHI  TCI  LST  SMI  pop  TWI  rain
year                                    
1995    0    0    0    0    1    8     6
2005    0    0    0    0    0    7     8
2015    0    0    0    0    0    6     8
2025    4    4    4    4    0    9    13


In [26]:
# Check the original condition-factor values for the affected OBJECTIDs
problem_ids = final.loc[
    final.isna().any(axis=1),
    "OBJECTID"
].dropna().unique()

problem_factors = factors[
    factors["OBJECTID"].isin(problem_ids)
].copy()

problem_factors[
    ["OBJECTID", "NDVI_1995", "TWI", "rain_1995",
     "VHI_2025", "TCI_2025", "LST_2025", "SMI_2025",
     "rain_2025"]
].sort_values("OBJECTID")

,OBJECTID,NDVI_1995,TWI,rain_1995,VHI_2025,TCI_2025,LST_2025,SMI_2025,rain_2025
50,51,0.752083,NaN,NaN,76.109360,0.700262,17.047847,0.698804,NaN
762,763,0.547744,NaN,1591.478027,50.834902,0.351596,30.965078,0.350873,1354.289063
1651,1652,0.551106,NaN,NaN,62.814145,0.666267,18.404797,0.664880,NaN
2543,2544,0.737784,NaN,1730.248047,76.247532,0.673032,18.134763,0.671631,1853.585815
3615,3616,0.574281,NaN,NaN,59.109273,0.534998,23.644464,0.533888,NaN
3816,3817,0.551835,1.900009,NaN,63.347906,0.671001,18.215829,0.669604,NaN
4193,4194,0.691308,NaN,NaN,72.795457,0.565250,22.436962,0.564076,NaN
4237,4238,0.545409,1.858702,NaN,61.901588,0.683617,17.712249,0.682194,NaN
4251,4252,0.493580,NaN,NaN,52.233007,0.534495,23.664541,0.533386,NaN
4376,4377,0.118741,1.378922,NaN,44.964694,0.743542,15.320331,0.741992,NaN


In [27]:
# Compare missing variables across all four years
check_ids = problem_ids

check = factors[factors["OBJECTID"].isin(check_ids)].copy()

cols_to_check = [
    "OBJECTID",
    "TWI",
    "rain_1995", "rain_2005", "rain_2015", "rain_2025",
    "VHI_1995", "VHI_2005", "VHI_2015", "VHI_2025",
    "TCI_1995", "TCI_2005", "TCI_2015", "TCI_2025",
    "LST_1995", "LST_2005", "LST_2015", "LST_2025",
    "SMI_1995", "SMI_2005", "SMI_2015", "SMI_2025"
]

check[cols_to_check].sort_values("OBJECTID")

,OBJECTID,TWI,rain_1995,rain_2005,rain_2015,rain_2025,VHI_1995,VHI_2005,VHI_2015,VHI_2025,...,TCI_2015,TCI_2025,LST_1995,LST_2005,LST_2015,LST_2025,SMI_1995,SMI_2005,SMI_2015,SMI_2025
50,51,NaN,NaN,NaN,NaN,NaN,79.200734,81.816677,73.348774,76.109360,...,0.675842,0.700262,13.204300,14.452343,17.967767,17.047847,0.785709,0.763718,0.675842,0.698804
762,763,NaN,1591.478027,1020.233826,1601.884766,1354.289063,49.276190,54.872740,38.439988,50.834902,...,0.301166,0.351596,27.570998,26.101153,32.952665,30.965078,0.404430,0.472480,0.301166,0.350873
1651,1652,NaN,NaN,NaN,NaN,NaN,61.519936,55.907362,56.025929,62.814145,...,0.580798,0.666267,18.478737,22.451515,21.768973,18.404797,0.645730,0.563727,0.580798,0.664880
2543,2544,NaN,1730.248047,1230.796143,1896.057983,1853.585815,77.878495,80.768762,73.399275,76.247532,...,0.652682,0.673032,13.627992,16.299498,18.894041,18.134763,0.774465,0.717536,0.652682,0.671631
3615,3616,NaN,NaN,NaN,NaN,NaN,62.595661,55.241501,51.958558,59.109273,...,0.508058,0.534998,18.596317,23.248344,24.678137,23.644464,0.642610,0.543805,0.508058,0.533888
3816,3817,1.900009,NaN,NaN,NaN,NaN,66.316781,56.489217,57.934333,63.347906,...,0.646666,0.671001,14.893012,20.441836,19.134649,18.215829,0.740892,0.613971,0.646666,0.669604
4193,4194,NaN,NaN,NaN,NaN,NaN,67.937784,72.059958,73.778618,72.795457,...,0.635092,0.565250,19.257814,23.194646,19.597526,22.436962,0.625054,0.545147,0.635092,0.564076
4237,4238,1.858702,NaN,NaN,NaN,NaN,69.582144,66.653509,61.603912,61.901588,...,0.747511,0.683617,12.174824,16.799862,15.101397,17.712249,0.813031,0.705026,0.747511,0.682194
4251,4252,NaN,NaN,NaN,NaN,NaN,62.605340,51.624522,48.225932,52.233007,...,0.494511,0.534495,15.356623,23.644997,25.219968,23.664541,0.728589,0.533888,0.494511,0.533386
4376,4377,1.378922,NaN,NaN,NaN,NaN,46.311789,42.422807,34.276445,44.964694,...,0.560737,0.743542,12.621790,14.596905,22.571292,15.320331,0.801169,0.760103,0.560737,0.741992


In [28]:
# Save the merged occurrence + condition-factor dataset

output_file = "PHSI_Master_Occurrence_ConditionFactors.csv"

final.to_csv(output_file, index=False)

print("Saved successfully:")
print(output_file)

print("\nRecords:", len(final))
print("Columns:", len(final.columns))
print("\nColumns:")
print(final.columns.tolist())

Saved successfully:
PHSI_Master_Occurrence_ConditionFactors.csv

Records: 136
Columns: 24

Columns:
['OBJECTID', 'Longitude', 'Latitude', 'year', 'TaxonName', 'NDVI', 'PCI', 'VHI', 'TCI', 'NDBI', 'NDMI', 'SAVI', 'LST', 'SMI', 'pop', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'LULC', 'rain', 'Presence']


In [29]:
# Verify the saved file
check = pd.read_csv("PHSI_Master_Occurrence_ConditionFactors.csv")

print("Records:", len(check))
print("Columns:", len(check.columns))
print("Missing values:", check.isna().sum().sum())

Records: 136
Columns: 24
Missing values: 82


In [30]:
import pandas as pd

# Load the saved master dataset
final = pd.read_csv("PHSI_Master_Occurrence_ConditionFactors.csv")

# Load the original condition-factor dataset
factors = pd.read_csv("Condition_Factors.csv")

# Standardize OBJECTID
final["OBJECTID"] = pd.to_numeric(final["OBJECTID"], errors="coerce").astype("Int64")
factors["OBJECTID"] = pd.to_numeric(factors["OBJECTID"], errors="coerce").astype("Int64")

years = [1995, 2005, 2015, 2025]

background_list = []

for year in years:

    # Occurrence locations for this particular year
    occurrence_ids = set(
        final.loc[final["year"] == year, "OBJECTID"].dropna()
    )

    # Number of occurrence records in this year
    n_occurrence = len(final[final["year"] == year])

    # We want 2 background records for every occurrence
    n_background = n_occurrence * 2

    # Candidate locations excluding occurrence locations
    candidates = factors[
        ~factors["OBJECTID"].isin(occurrence_ids)
    ].copy()

    # Randomly select background locations
    background = candidates.sample(
        n=n_background,
        random_state=42 + year
    ).copy()

    # Add modelling information
    background["year"] = year
    background["TaxonName"] = "Background"
    background["Presence"] = 0

    background_list.append(background)

# Combine background data
background_all = pd.concat(
    background_list,
    ignore_index=True
)

print("Background records:", len(background_all))

print("\nBackground records by year:")
print(background_all.groupby("year").size())

Background records: 272

Background records by year:
year
1995     50
2005     74
2015     44
2025    104
dtype: int64


In [31]:
# Create the same 24-column structure for background records

background_final = background_all.copy()

# Create year-specific environmental variables
dynamic_variables = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "rain"
]

for var in dynamic_variables:
    background_final[var] = pd.NA
    
    for year in years:
        source_column = f"{var}_{year}"
        mask = background_final["year"] == year
        
        background_final.loc[mask, var] = (
            background_final.loc[mask, source_column]
        )

# Create year-specific LULC
background_final["LULC"] = pd.NA

for year in years:
    source_column = f"Nyeri_LULC_{year}"
    mask = background_final["year"] == year
    
    background_final.loc[mask, "LULC"] = (
        background_final.loc[mask, source_column]
    )

# Select exactly the same columns as the occurrence dataset
background_final = background_final[[
    "OBJECTID",
    "Longitude",
    "Latitude",
    "year",
    "TaxonName",
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI",
    "LULC",
    "rain",
    "Presence"
]]

print("Background records:", len(background_final))
print("Background columns:", len(background_final.columns))
print("\nColumns:")
print(background_final.columns.tolist())

Background records: 272
Background columns: 24

Columns:
['OBJECTID', 'Longitude', 'Latitude', 'year', 'TaxonName', 'NDVI', 'PCI', 'VHI', 'TCI', 'NDBI', 'NDMI', 'SAVI', 'LST', 'SMI', 'pop', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'LULC', 'rain', 'Presence']


In [32]:
print(background_final.head())

print("\nPresence values:")
print(background_final["Presence"].value_counts())

print("\nBackground records by year:")
print(background_final.groupby("year").size())

   OBJECTID  Longitude  Latitude  year   TaxonName NDVI  PCI  VHI  TCI NDBI  \
0      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
1      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
2      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
3      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
4      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   

   ...  pop SOC Aspect Slope  PH  TRI  TWI  LULC  rain  Presence  
0  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
1  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
2  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
3  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
4  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  

[5 rows x 24 columns]

Presence values:
Presence
0    272
Name: count, dtype: int64

Background records by year:
year
1995

In [33]:
print(background_final.head())

print("\nMissing values:")
print(background_final.isna().sum())

print("\nPresence:")
print(background_final["Presence"].value_counts())

print("\nRecords by year:")
print(background_final.groupby("year").size())

   OBJECTID  Longitude  Latitude  year   TaxonName NDVI  PCI  VHI  TCI NDBI  \
0      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
1      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
2      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
3      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   
4      <NA>        NaN       NaN  1995  Background  NaN  NaN  0.0  NaN  0.0   

   ...  pop SOC Aspect Slope  PH  TRI  TWI  LULC  rain  Presence  
0  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
1  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
2  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
3  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  
4  ...  NaN NaN    NaN   NaN NaN  NaN  NaN   NaN   NaN         0  

[5 rows x 24 columns]

Missing values:
OBJECTID     252
Longitude    252
Latitude     252
year           0
TaxonName      

In [34]:
print("Condition-factor records:", len(factors))

print("\nFirst 5 OBJECTIDs:")
print(factors["OBJECTID"].head())

print("\nOBJECTID data type:")
print(factors["OBJECTID"].dtype)

print("\nMissing OBJECTIDs:")
print(factors["OBJECTID"].isna().sum())

print("\nFirst 5 rows:")
print(factors.head())

Condition-factor records: 65535

First 5 OBJECTIDs:
0    1
1    2
2    3
3    4
4    5
Name: OBJECTID, dtype: Int64

OBJECTID data type:
Int64

Missing OBJECTIDs:
60535

First 5 rows:
   OBJECTID  Longitude  Latitude  NDVI_1995  NDVI_2005  NDVI_2015  NDVI_2025  \
0         1  36.833259 -0.397848   0.770591   0.806752   0.837014   0.671456   
1         2  36.794072 -0.296912   0.520596   0.493491   0.511073   0.549253   
2         3  37.144445 -0.447023   0.638068   0.562411   0.455919   0.735100   
3         4  36.674651 -0.485149   0.583268   0.549556   0.556273   0.643500   
4         5  37.052083 -0.146314   0.523536   0.376470   0.389503   0.640413   

    PCI_1995   PCI_2005   PCI_2015  ...       TRI       TWI  Nyeri_LULC_1995  \
0  40.618143  13.666149  43.679842  ...  0.691033  3.340299              1.0   
1  39.798664  13.459880  42.701731  ...  1.146779  3.090171              2.0   
2  42.097407  15.304171  48.914979  ...  0.164172  4.311491              4.0   
3  39.554880  1

In [35]:
# Keep only valid spatial locations
factors_valid = factors[
    factors["OBJECTID"].notna()
].copy()

print("Valid condition-factor locations:", len(factors_valid))

print("\nFirst 5 OBJECTIDs:")
print(factors_valid["OBJECTID"].head())

print("\nLast 5 OBJECTIDs:")
print(factors_valid["OBJECTID"].tail())

print("\nMissing OBJECTIDs:")
print(factors_valid["OBJECTID"].isna().sum())

Valid condition-factor locations: 5000

First 5 OBJECTIDs:
0    1
1    2
2    3
3    4
4    5
Name: OBJECTID, dtype: Int64

Last 5 OBJECTIDs:
4995    4996
4996    4997
4997    4998
4998    4999
4999    5000
Name: OBJECTID, dtype: Int64

Missing OBJECTIDs:
0


In [36]:
years = [1995, 2005, 2015, 2025]

background_list = []

for year in years:

    # Occurrence locations for this year
    occurrence_ids = set(
        final.loc[
            final["year"] == year,
            "OBJECTID"
        ].dropna()
    )

    # Number of occurrence records
    n_occurrence = len(
        final[final["year"] == year]
    )

    # Two background records for every occurrence
    n_background = n_occurrence * 2

    # Valid candidate locations only
    candidates = factors_valid[
        ~factors_valid["OBJECTID"].isin(occurrence_ids)
    ].copy()

    # Randomly select background locations
    background = candidates.sample(
        n=n_background,
        random_state=42 + year
    ).copy()

    # Add modelling information
    background["year"] = year
    background["TaxonName"] = "Background"
    background["Presence"] = 0

    background_list.append(background)

# Combine all years
background_all = pd.concat(
    background_list,
    ignore_index=True
)

print("Background records:", len(background_all))

print("\nBackground records by year:")
print(background_all.groupby("year").size())

Background records: 272

Background records by year:
year
1995     50
2005     74
2015     44
2025    104
dtype: int64


In [37]:
print(background_all[
    ["OBJECTID", "Longitude", "Latitude", "year", "TaxonName", "Presence"]
].head(10))

print("\nMissing values in key fields:")
print(
    background_all[
        ["OBJECTID", "Longitude", "Latitude"]
    ].isna().sum()
)

   OBJECTID  Longitude  Latitude  year   TaxonName  Presence
0       525  36.941580 -0.551512  1995  Background         0
1      4443  37.166911 -0.432487  1995  Background         0
2      4452  37.177485 -0.423285  1995  Background         0
3       522  37.112742 -0.517694  1995  Background         0
4      1648  36.795565 -0.207470  1995  Background         0
5      3567  37.145821 -0.092390  1995  Background         0
6      1114  37.141734 -0.216899  1995  Background         0
7      2267  36.684756 -0.349376  1995  Background         0
8      4929  37.055402 -0.060117  1995  Background         0
9       306  36.778305 -0.123796  1995  Background         0

Missing values in key fields:
OBJECTID     0
Longitude    0
Latitude     0
dtype: int64


In [38]:
dynamic_variables = [
    "NDVI", "PCI", "VHI", "TCI", "NDBI",
    "NDMI", "SAVI", "LST", "SMI", "pop", "rain"
]

background_final_list = []

for year in years:

    # Select background records for this year
    bg = background_all[
        background_all["year"] == year
    ].copy()

    # Create the year-specific predictor columns
    for var in dynamic_variables:
        bg[var] = bg[f"{var}_{year}"]

    # Year-specific LULC
    bg["LULC"] = bg[f"Nyeri_LULC_{year}"]

    background_final_list.append(bg)

# Combine all years
background_final = pd.concat(
    background_final_list,
    ignore_index=True
)

# Keep exactly the same structure as the occurrence dataset
background_final = background_final[
    [
        "OBJECTID",
        "Longitude",
        "Latitude",
        "year",
        "TaxonName",
        "NDVI",
        "PCI",
        "VHI",
        "TCI",
        "NDBI",
        "NDMI",
        "SAVI",
        "LST",
        "SMI",
        "pop",
        "SOC",
        "Aspect",
        "Slope",
        "PH",
        "TRI",
        "TWI",
        "LULC",
        "rain",
        "Presence"
    ]
].copy()

print("Background records:", len(background_final))
print("Background columns:", len(background_final.columns))

print("\nFirst 5 background records:")
print(background_final.head())

Background records: 272
Background columns: 24

First 5 background records:
   OBJECTID  Longitude  Latitude  year   TaxonName      NDVI        PCI  \
0       525  36.941580 -0.551512  1995  Background  0.636007  40.723491   
1      4443  37.166911 -0.432487  1995  Background  0.663094  42.167328   
2      4452  37.177485 -0.423285  1995  Background  0.665279  42.191655   
3       522  37.112742 -0.517694  1995  Background  0.405234  41.652707   
4      1648  36.795565 -0.207470  1995  Background  0.525299  38.881251   

         VHI       TCI      NDBI  ...          pop        SOC      Aspect  \
0  56.730903  0.459701  0.229289  ...  1619.729641  33.075653  287.402069   
1  60.241271  0.501115  0.218419  ...   345.195834  36.085098   58.845901   
2  60.936144  0.512690  0.210465  ...   234.948538  42.000000  116.603195   
3  41.181042  0.394013 -0.014280  ...   510.682755  34.000000  175.167114   
4  47.354597  0.389857 -0.004861  ...    50.645267  23.828167  151.370056   

      Slop

In [39]:
print("\nMissing values:")
print(background_final.isna().sum())

print("\nRecords by year:")
print(background_final.groupby("year").size())

print("\nPresence:")
print(background_final["Presence"].value_counts())


Missing values:
OBJECTID      0
Longitude     0
Latitude      0
year          0
TaxonName     0
NDVI          1
PCI           0
VHI           1
TCI           1
NDBI          0
NDMI          1
SAVI          1
LST           1
SMI           1
pop           0
SOC           0
Aspect        0
Slope         0
PH            0
TRI           0
TWI           2
LULC          1
rain         12
Presence      0
dtype: int64

Records by year:
year
1995     50
2005     74
2015     44
2025    104
dtype: int64

Presence:
Presence
0    272
Name: count, dtype: int64


In [40]:
# Combine occurrence and background records
phsi_model_data = pd.concat(
    [final, background_final],
    ignore_index=True
)

print("Total modelling records:", len(phsi_model_data))
print("Total columns:", len(phsi_model_data.columns))

print("\nRecords by Presence:")
print(phsi_model_data["Presence"].value_counts())

print("\nRecords by year and Presence:")
print(
    phsi_model_data.groupby(
        ["year", "Presence"]
    ).size()
)

Total modelling records: 408
Total columns: 24

Records by Presence:
Presence
0    272
1    136
Name: count, dtype: int64

Records by year and Presence:
year  Presence
1995  0            50
      1            25
2005  0            74
      1            37
2015  0            44
      1            22
2025  0           104
      1            52
dtype: int64


In [41]:
# Check duplicate OBJECTIDs within each year
duplicates = (
    phsi_model_data
    .groupby(["year", "OBJECTID"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate locations within the same year:")
print(len(duplicates))

# Check whether any background location overlaps an occurrence
occurrence_locations = set(
    zip(
        phsi_model_data.loc[
            phsi_model_data["Presence"] == 1, "year"
        ],
        phsi_model_data.loc[
            phsi_model_data["Presence"] == 1, "OBJECTID"
        ]
    )
)

background_locations = set(
    zip(
        phsi_model_data.loc[
            phsi_model_data["Presence"] == 0, "year"
        ],
        phsi_model_data.loc[
            phsi_model_data["Presence"] == 0, "OBJECTID"
        ]
    )
)

overlap = occurrence_locations.intersection(
    background_locations
)

print("\nOccurrence-background overlaps:")
print(len(overlap))

Duplicate locations within the same year:
31

Occurrence-background overlaps:
0


In [42]:
# Show the 31 duplicated year-location combinations
duplicate_details = phsi_model_data[
    phsi_model_data.duplicated(
        subset=["year", "OBJECTID"],
        keep=False
    )
].sort_values(
    ["year", "OBJECTID", "Presence", "TaxonName"]
)

print(duplicate_details[
    ["OBJECTID", "Longitude", "Latitude",
     "year", "TaxonName", "Presence"]
].to_string(index=False))

 OBJECTID  Longitude  Latitude  year                                                                   TaxonName  Presence
      597  37.130871 -0.131246  1995         Aneleus politus (Santschi, 1914) = Carebara polita (Santschi, 1914)         1
      597  37.130871 -0.131246  1995     Oligomyrmex politus (Santschi, 1914) = Carebara polita (Santschi, 1914)         1
      763  37.000000 -0.633333  1995                                                         polyrhachis viscosa         1
      763  37.000000 -0.633330  1995                                                         tetraponera ambigua         1
      763  37.000000 -0.633330  1995                                                          tetraponera gerdae         1
      763  37.000000 -0.633330  1995                                                          tetraponera parops         1
     1795  36.831490 -0.347810  1995                                                           discothyrea dryad         1
     1795  36.83

In [43]:
background_duplicates = (
    background_final
    .groupby(["year", "OBJECTID"])
    .size()
    .reset_index(name="count")
)

background_duplicates = background_duplicates[
    background_duplicates["count"] > 1
]

print("Duplicate background locations within the same year:")
print(len(background_duplicates))

if len(background_duplicates) > 0:
    print("\nDuplicated background locations:")
    print(background_duplicates)

Duplicate background locations within the same year:
0


In [44]:
output_file = "PHSI_Final_Modelling_Dataset.csv"

phsi_model_data.to_csv(
    output_file,
    index=False
)

print("Saved successfully:")
print(output_file)

print("\nFinal dataset shape:")
print(phsi_model_data.shape)

Saved successfully:
PHSI_Final_Modelling_Dataset.csv

Final dataset shape:
(408, 24)


In [45]:
import pandas as pd

# Load datasets
conditions = pd.read_csv("Condition_Factors.csv")
occurrence = pd.read_csv("Occurance.csv")

# -------------------------------------------------------
# 1. Keep only the 5,000 valid spatial locations
# -------------------------------------------------------

conditions["OBJECTID"] = pd.to_numeric(
    conditions["OBJECTID"], errors="coerce"
).astype("Int64")

factors_valid = conditions[
    conditions["OBJECTID"].notna()
].copy()

print("Valid spatial locations:", len(factors_valid))


# -------------------------------------------------------
# 2. Define the four modelling years
# -------------------------------------------------------

years = [1995, 2005, 2015, 2025]


# -------------------------------------------------------
# 3. Variables that change by year
# -------------------------------------------------------

dynamic_variables = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "rain"
]


# -------------------------------------------------------
# 4. Variables that remain constant
# -------------------------------------------------------

static_variables = [
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI"
]


# -------------------------------------------------------
# 5. Create 5,000 records for each year
# -------------------------------------------------------

yearly_data = []

for year in years:

    df = factors_valid[
        ["OBJECTID", "Longitude", "Latitude"]
        + static_variables
    ].copy()

    # Add year
    df["year"] = year

    # Add year-specific dynamic variables
    for var in dynamic_variables:
        df[var] = factors_valid[f"{var}_{year}"]

    # Add year-specific LULC
    df["LULC"] = factors_valid[f"Nyeri_LULC_{year}"]

    yearly_data.append(df)


# Combine all four years
model_data = pd.concat(
    yearly_data,
    ignore_index=True
)


# -------------------------------------------------------
# 6. Add Presence information
# -------------------------------------------------------

occurrence["OBJECTID"] = pd.to_numeric(
    occurrence["OBJECTID"], errors="coerce"
).astype("Int64")

# Start with background = 0
model_data["Presence"] = 0


# Mark locations with occurrence records as presence = 1
occurrence_keys = set(
    zip(
        occurrence["OBJECTID"],
        occurrence["year"]
    )
)

model_data["Presence"] = [
    1 if (oid, year) in occurrence_keys else 0
    for oid, year in zip(
        model_data["OBJECTID"],
        model_data["year"]
    )
]


# -------------------------------------------------------
# 7. Check the result
# -------------------------------------------------------

print("\nFinal dataset shape:")
print(model_data.shape)

print("\nRecords by year:")
print(model_data["year"].value_counts().sort_index())

print("\nPresence counts:")
print(model_data["Presence"].value_counts())

print("\nPresence by year:")
print(
    model_data.groupby(
        ["year", "Presence"]
    ).size()
)

Valid spatial locations: 5000

Final dataset shape:
(20000, 23)

Records by year:
year
1995    5000
2005    5000
2015    5000
2025    5000
Name: count, dtype: int64

Presence counts:
Presence
0    19912
1       88
Name: count, dtype: int64

Presence by year:
year  Presence
1995  0           4984
      1             16
2005  0           4976
      1             24
2015  0           4983
      1             17
2025  0           4969
      1             31
dtype: int64


In [46]:
# Save the complete 20,000-record modelling dataset

output_file = "PHSI_Complete_5000Locations_4Years.csv"

model_data.to_csv(
    output_file,
    index=False
)

print("Saved successfully:")
print(output_file)

print("\nDataset shape:")
print(model_data.shape)

Saved successfully:
PHSI_Complete_5000Locations_4Years.csv

Dataset shape:
(20000, 23)


In [47]:
# Check missing values in every variable

missing_values = model_data.isna().sum()

print("Missing values by column:")
print(missing_values[missing_values > 0])

Missing values by column:
Aspect      20
Slope       20
PH           4
TWI        316
NDVI        18
PCI          3
VHI        128
TCI        147
NDMI        18
SAVI        18
LST        147
SMI        147
pop         26
rain      1100
LULC        18
dtype: int64


In [48]:
# Check missing values by year

missing_by_year = (
    model_data
    .groupby("year")
    .apply(lambda x: x.isna().sum(), include_groups=False)
)

print(missing_by_year)

      OBJECTID  Longitude  Latitude  SOC  Aspect  Slope  PH  TRI  TWI  NDVI  \
year                                                                          
1995         0          0         0    0       5      5   1    0   79     1   
2005         0          0         0    0       5      5   1    0   79    12   
2015         0          0         0    0       5      5   1    0   79     1   
2025         0          0         0    0       5      5   1    0   79     4   

      ...  TCI  NDBI  NDMI  SAVI  LST  SMI  pop  rain  LULC  Presence  
year  ...                                                              
1995  ...   22     0     1     1   22   22   13   275     1         0  
2005  ...   56     0    12    12   56   56    0   275    12         0  
2015  ...   41     0     1     1   41   41    0   275     1         0  
2025  ...   28     0     4     4   28   28   13   275     4         0  

[4 rows x 22 columns]


In [49]:
# Check missing values by year for the main variables

variables_to_check = [
    "Aspect",
    "Slope",
    "PH",
    "TWI",
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "rain",
    "LULC"
]

for year in [1995, 2005, 2015, 2025]:

    print(f"\n===== {year} =====")

    print(
        model_data.loc[
            model_data["year"] == year,
            variables_to_check
        ].isna().sum()
    )


===== 1995 =====
Aspect      5
Slope       5
PH          1
TWI        79
NDVI        1
PCI         1
VHI         0
TCI        22
NDMI        1
SAVI        1
LST        22
SMI        22
pop        13
rain      275
LULC        1
dtype: int64

===== 2005 =====
Aspect      5
Slope       5
PH          1
TWI        79
NDVI       12
PCI         0
VHI        62
TCI        56
NDMI       12
SAVI       12
LST        56
SMI        56
pop         0
rain      275
LULC       12
dtype: int64

===== 2015 =====
Aspect      5
Slope       5
PH          1
TWI        79
NDVI        1
PCI         1
VHI        39
TCI        41
NDMI        1
SAVI        1
LST        41
SMI        41
pop         0
rain      275
LULC        1
dtype: int64

===== 2025 =====
Aspect      5
Slope       5
PH          1
TWI        79
NDVI        4
PCI         1
VHI        27
TCI        28
NDMI        4
SAVI        4
LST        28
SMI        28
pop        13
rain      275
LULC        4
dtype: int64


In [50]:
# Check missing predictor values specifically at presence locations

predictors = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI",
    "LULC",
    "rain"
]

presence_data = model_data[
    model_data["Presence"] == 1
].copy()

print("Number of presence records:", len(presence_data))

print("\nMissing values among presence records:")
print(
    presence_data[predictors]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Number of presence records: 88

Missing values among presence records:
rain      23
TWI       18
pop        1
LST        1
VHI        1
TCI        1
SMI        1
NDVI       0
PCI        0
NDBI       0
SAVI       0
NDMI       0
SOC        0
Aspect     0
PH         0
Slope      0
TRI        0
LULC       0
dtype: int64


In [51]:
# Show the actual presence records that contain missing predictors

presence_missing = presence_data[
    presence_data[predictors].isna().any(axis=1)
]

print("Presence records with at least one missing predictor:")
print(len(presence_missing))

print("\nAffected presence records:")
print(
    presence_missing[
        ["OBJECTID", "Longitude", "Latitude", "year", "TaxonName"]
        + predictors
    ]
)

Presence records with at least one missing predictor:
27

Affected presence records:


KeyError: "['TaxonName'] not in index"

In [52]:
# Show presence locations with missing predictor values

presence_missing = presence_data[
    presence_data[predictors].isna().any(axis=1)
].copy()

print("Presence records with at least one missing predictor:")
print(len(presence_missing))

print("\nAffected presence records:")
print(
    presence_missing[
        ["OBJECTID", "Longitude", "Latitude", "year"]
        + predictors
    ].to_string(index=False)
)

Presence records with at least one missing predictor:
27

Affected presence records:
 OBJECTID  Longitude  Latitude  year     NDVI       PCI       VHI      TCI      NDBI      NDMI     SAVI       LST      SMI         pop       SOC     Aspect     Slope       PH      TRI      TWI  LULC        rain
      763  37.022924 -0.588006  1995 0.547744 40.565070 49.276190 0.404430  0.190476  0.190476 0.300664 27.570998 0.404430  360.981494 42.948906 204.373871 13.032513 5.407969 0.623312      NaN   4.0 1591.478027
     2544  36.665354 -0.534494  1995 0.737784 39.282827 77.878495 0.774465  0.190369  0.190369 0.388026 13.627992 0.774465    0.000000 64.713127 115.625153  7.896955 5.263845 0.785643      NaN   0.0 1730.248047
     4252  37.266493 -0.252522  1995 0.493580 41.162658 62.605340 0.728589  0.048044  0.048044 0.292732 15.356623 0.728589    0.000680 82.973686 171.246597 24.025703 5.324329 0.218313      NaN   2.0         NaN
     4377  37.298049 -0.148525  1995 0.118741 39.568740 46.311789 0.801

In [53]:
# ============================================================
# STEP 5: CHECK DATA TYPES AND PREDICTOR STRUCTURE
# ============================================================

predictors = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI",
    "LULC",
    "rain"
]

print("DATA TYPES")
print("=" * 50)

print(model_data[predictors].dtypes)


print("\n\nUNIQUE LULC VALUES")
print("=" * 50)

print(
    model_data["LULC"]
    .value_counts(dropna=False)
    .sort_index()
)


print("\n\nTARGET VARIABLE")
print("=" * 50)

print(model_data["Presence"].value_counts())


print("\n\nYEAR")
print("=" * 50)

print(model_data["year"].value_counts().sort_index())

DATA TYPES
NDVI      float64
PCI       float64
VHI       float64
TCI       float64
NDBI      float64
NDMI      float64
SAVI      float64
LST       float64
SMI       float64
pop       float64
SOC       float64
Aspect    float64
Slope     float64
PH        float64
TRI       float64
TWI       float64
LULC      float64
rain      float64
dtype: object


UNIQUE LULC VALUES
LULC
0.0    5227
1.0    2875
2.0    3530
3.0    4334
4.0    4016
NaN      18
Name: count, dtype: int64


TARGET VARIABLE
Presence
0    19912
1       88
Name: count, dtype: int64


YEAR
year
1995    5000
2005    5000
2015    5000
2025    5000
Name: count, dtype: int64


In [54]:
# ============================================================
# STEP 6: PREPARE MODEL DATA
# ============================================================

import pandas as pd

# Predictor variables
predictors = [
    "NDVI",
    "PCI",
    "VHI",
    "TCI",
    "NDBI",
    "NDMI",
    "SAVI",
    "LST",
    "SMI",
    "pop",
    "SOC",
    "Aspect",
    "Slope",
    "PH",
    "TRI",
    "TWI",
    "LULC",
    "rain"
]

# Create modelling dataframe
X = model_data[predictors].copy()

# Target
y = model_data["Presence"].copy()

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nPredictors:")
print(list(X.columns))

print("\nTarget distribution:")
print(y.value_counts())

Predictor matrix shape: (20000, 18)
Target shape: (20000,)

Predictors:
['NDVI', 'PCI', 'VHI', 'TCI', 'NDBI', 'NDMI', 'SAVI', 'LST', 'SMI', 'pop', 'SOC', 'Aspect', 'Slope', 'PH', 'TRI', 'TWI', 'LULC', 'rain']

Target distribution:
Presence
0    19912
1       88
Name: count, dtype: int64


In [55]:
# Check whether XGBoost is installed

try:
    import xgboost as xgb
    print("XGBoost is installed.")
    print("Version:", xgb.__version__)
except ImportError:
    print("XGBoost is NOT installed.")

XGBoost is NOT installed.
